# ZK-FL StyleGAN3-r Retina — Colab Runner

Bu notebook iş mantığı İÇERMEZ. Her hücre repodan bir fonksiyon/script import edip çağırır. Mod seçimine göre Faz A adımları (`inventory_dry`, `inventory`, `extract`, `audit`) ya da ileri fazlar çalışır.

## 1. Drive mount

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. GitHub clone/pull

PAT, Colab Secrets'tan `GH_PAT` adıyla okunur. PAT hiçbir zaman bu notebook'a yazılmaz.

In [ ]:
import os
from google.colab import userdata

GH_PAT = userdata.get("GH_PAT")
REPO_URL = "github.com/<kullanici>/zk-fl-stylegan-retina.git"
REPO_DIR = "/content/zk-fl-stylegan-retina"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://{GH_PAT}@{REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 3. StyleGAN-XL klonu

`network-snapshot.pkl` dosyaları `legacy.load_network_pkl` ile açılabilmesi için `dnnlib`/`torch_utils`/`legacy.py`'ye ihtiyaç duyar. Yol `configs/paths.yaml: colab.stylegan_xl_repo`'dan okunur (varsayılan `/content/stylegan-xl`).

In [ ]:
import yaml

with open("configs/paths.yaml") as f:
    _paths = yaml.safe_load(f)["colab"]

STYLEGAN_XL_REPO = _paths["stylegan_xl_repo"]

if not os.path.isdir(STYLEGAN_XL_REPO):
    !git clone https://github.com/autonomousvision/stylegan-xl {STYLEGAN_XL_REPO}
else:
    print(f"StyleGAN-XL zaten mevcut: {STYLEGAN_XL_REPO}")

## 4. Bağımlılık kurulumu

In [ ]:
!bash scripts/setup_colab.sh

## 5. Mod seçimi

Faz A önerilen sıra: `inventory_dry` (dosya açmadan tarama, önce bunu çalıştır) → `inventory` (gerçek pkl analizi) → `extract` (shard çıkarımı) → `audit` (fedavg denetimi). Her modu ayrı ayrı, MODE'u değiştirip bu hücreden sonrasını yeniden çalıştırarak sırayla koştur.

In [ ]:
# inventory_dry | inventory | extract | audit | replay | attack | bench | live_round
MODE = "inventory_dry"

## 6. Modu çalıştır

Her dal, ilgili `scripts/*.py` modülünü `python -m` ile çağırır. İş mantığı burada değil, o modüllerde/`fl/` paketinde yaşar.

In [ ]:
if MODE == "inventory_dry":
    !python -m scripts.inventory --env colab --dry-run
elif MODE == "inventory":
    !python -m scripts.inventory --env colab
elif MODE == "extract":
    !python -m scripts.extract_shards --env colab
elif MODE == "audit":
    !python -m scripts.audit_fedavg --env colab
elif MODE == "replay":
    raise NotImplementedError("scripts/replay_proofs.py henuz yazilmadi (Faz E)")
elif MODE == "attack":
    raise NotImplementedError("attacks/ henuz yazilmadi (Faz F)")
elif MODE == "bench":
    raise NotImplementedError("scripts/bench_circuit.py henuz yazilmadi (Faz C)")
elif MODE == "live_round":
    raise NotImplementedError("orchestrator/ henuz yazilmadi (Faz D/F)")
else:
    raise ValueError(f"Bilinmeyen MODE: {MODE}")

## 7. Sonuçları git ile geri gönder

Sadece küçük sonuç JSON'ları (`zk_artifacts_results/`) commit edilir. Ağırlık dosyaları/shard'lar (`zk_artifacts/`, Drive'da) ve `.gitignore` ile dışlanan her şey repoya girmez.

In [ ]:
!git add zk_artifacts_results/
!git commit -m "Colab: {MODE} sonuclari"
!git push